# SBA Featurization Notebook

This notebook converts the canonical SBA training, validation, and test splits into model-ready numeric datasets.

The feature universe is intentionally limited to the five highest-ranked features in the reference `feature_screener_sba.ipynb` report: `bankname`, `bankcity`, `initialinterestrate`, `bankstreet`, and `terminmonths`.

All learned transformations are fit on training data only. Validation and test rows are transformed with frozen training artifacts.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from category_encoders import MEstimateEncoder
from sklearn.impute import SimpleImputer

TARGET_COL = 'loan_status_r'
PARTITION_COL = 'dataset_partition'

# These are the first five rows of the screener's ranked report, in rank order.
NUMERIC_INPUT_FEATURES = [
    'initialinterestrate',
    'terminmonths',
]
CATEGORICAL_INPUT_FEATURES = [
    'bankname',
    'bankcity',
    'bankstreet',
]
TOP_FIVE_FEATURES = [
    'bankname',
    'bankcity',
    'initialinterestrate',
    'bankstreet',
    'terminmonths',
]

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [working_directory, *working_directory.parents]
        if (candidate / 'data' / 'dd_cleaner' / 'sba_user_cleaned_raw_training.csv').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate the project root from current working directory: {working_directory}'
    )

RAW_SPLIT_DIR = PROJECT_ROOT / 'data' / 'dd_cleaner'
OUT_DIR = PROJECT_ROOT / 'data' / 'featurization'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Top-five ranked raw features:', TOP_FIVE_FEATURES)


## 1) Load and validate canonical splits

The persisted split files are authoritative. This notebook does not create a second random split.


In [ ]:
split_files = {
    'train': 'sba_user_cleaned_raw_training.csv',
    'validation': 'sba_user_cleaned_raw_validation.csv',
    'test': 'sba_user_cleaned_raw_test.csv',
}

frames = {}
required_columns = set(TOP_FIVE_FEATURES + [TARGET_COL])
for partition, file_name in split_files.items():
    path = RAW_SPLIT_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f'Missing canonical {partition} split: {path}')
    frame = pd.read_csv(path)
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise ValueError(f'{partition} is missing required columns: {missing}')
    if frame[TARGET_COL].isna().any():
        raise ValueError(f'{partition} has missing target values in {TARGET_COL}')
    frame = frame[TOP_FIVE_FEATURES + [TARGET_COL]].copy()
    frame[PARTITION_COL] = partition
    frames[partition] = frame
    print(f'{partition}: rows={len(frame)}, target_counts={frame[TARGET_COL].value_counts().to_dict()}')

train_df = frames['train']
validation_df = frames['validation']
test_df = frames['test']


## 2) Numeric preprocessing

Median values are learned from the training split only and reused unchanged for validation and test.


In [ ]:
numeric_imputer = SimpleImputer(strategy='median')
numeric_imputer.fit(train_df[NUMERIC_INPUT_FEATURES])

numeric_frames = {}
for partition, frame in frames.items():
    numeric_frames[partition] = pd.DataFrame(
        numeric_imputer.transform(frame[NUMERIC_INPUT_FEATURES]),
        columns=NUMERIC_INPUT_FEATURES,
        index=frame.index,
    )
    print(f'{partition} numeric shape: {numeric_frames[partition].shape}')


## 3) M-estimate categorical encoding

The M-estimate target encoder is fit only on training categorical values and training labels, then applied without refitting to validation and test.


In [ ]:
categorical_encoder = MEstimateEncoder(
    cols=CATEGORICAL_INPUT_FEATURES,
    m=10,
    handle_unknown='value',
    handle_missing='value',
)
categorical_encoder.fit(
    train_df[CATEGORICAL_INPUT_FEATURES],
    train_df[TARGET_COL],
)

categorical_frames = {}
for partition, frame in frames.items():
    categorical_frames[partition] = categorical_encoder.transform(
        frame[CATEGORICAL_INPUT_FEATURES]
    ).astype(float)
    print(f'{partition} categorical shape: {categorical_frames[partition].shape}')


## 4) Final numeric export

The model-ready feature contract contains exactly the five requested screener features, plus the target and authoritative partition label. No geographic or clustering-derived columns are included because they are outside the top-five ranked feature set.


In [ ]:
exports = {}
for partition, frame in frames.items():
    encoded = pd.concat(
        [
            numeric_frames[partition],
            categorical_frames[partition],
        ],
        axis=1,
    )
    # Preserve the reference notebook's top-five display order.
    encoded = encoded[TOP_FIVE_FEATURES]
    encoded[TARGET_COL] = frame[TARGET_COL].astype(int)
    encoded[PARTITION_COL] = partition
    exports[partition] = encoded

    path = OUT_DIR / f'{partition}_numeric_data.csv'
    encoded.to_csv(path, index=False)
    print(f'Saved {path}: {encoded.shape}')

combined = pd.concat(
    [exports['train'], exports['validation'], exports['test']],
    ignore_index=True,
)
combined_path = OUT_DIR / 'model_ready_numeric_data.csv'
combined.to_csv(combined_path, index=False)

joblib.dump(
    {
        'top_five_features': TOP_FIVE_FEATURES,
        'numeric_features': NUMERIC_INPUT_FEATURES,
        'categorical_features': CATEGORICAL_INPUT_FEATURES,
        'numeric_imputer': numeric_imputer,
        'categorical_encoder': categorical_encoder,
        'target_column': TARGET_COL,
        'partition_column': PARTITION_COL,
    },
    OUT_DIR / 'sba_featurization_bundle.joblib',
)

print(f'Saved combined file: {combined_path}: {combined.shape}')
print('partition counts:', combined[PARTITION_COL].value_counts().to_dict())
print('target counts:', combined[TARGET_COL].value_counts().to_dict())
print('final columns:', combined.columns.tolist())
